In [ ]:
import numpy as np
import random
import string

from collections import namedtuple

In [ ]:
psi = np.array([[1, 0], [0, 0]])
h = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
cp = np.diag([1, 1, 1, 1j]).reshape((2, 2, 2, 2))

In [ ]:
# np.einsum("ij,ip,pjqr,rs->qs", psi, h, cp, h)
psi1 = np.einsum("ij,ip->jp", psi, h)
psi2 = np.einsum("pj,jpqr->qr", psi1, cp)
psi3 = np.einsum("qr,rs->qs", psi2, h)

print(psi1)
print(psi2)
print(psi3)

In [ ]:
t1 = np.array([[1, 2], [3, 4]])
t2 = np.array([[1, 2], [3, 4]])

np.einsum("ij,ki->jk", t1, t2)

In [ ]:
def random_tensor(dims, dp = 1):
  real = (np.random.randint(2 * 10**dp, size = dims) - 10**dp) / 10**dp
  imag = (np.random.randint(2 * 10**dp, size = dims) - 10**dp) / 10**dp

  return real + imag * 1j

t1 = random_tensor((4, 5))
t2 = random_tensor((3, 5, 7))

In [ ]:
Contraction = namedtuple("Contraction", "t1 t2 t3 ws")
c1 = Contraction(t1, t2, np.einsum("ij,kjl->ikl", t1, t2), [(0, 1)])

for el in c1.t3.flatten():
  plus = "+" if el.imag >= 0 else ""
  print(f"{el.real: .2f}{plus}{el.imag:.2f}i")

print(c1.t3)

In [ ]:
def make_contraction(t1, t2, ws):
  letters = list(string.ascii_lowercase)

  t1_idxs = letters[0:len(t1.shape)]
  t2_idxs = letters[len(t1.shape):(len(t1.shape) + len(t2.shape))]

  for i1, i2 in ws:
    t2_idxs[i2] = t1_idxs[i1]

  con_string = ''.join(t1_idxs) + ',' + ''.join(t2_idxs)

  return Contraction(t1, t2, np.einsum(con_string, t1, t2), ws)


np.random.seed(9457)

t1 = random_tensor((2, 2))
t2 = random_tensor((2, 2))

make_contraction(t1, t2, [(0, 1)])

In [ ]:
def random_dims_and_wires(max_idxs, allowed_dims):
  n1 = np.random.randint(1, max_idxs)
  n2 = np.random.randint(1, max_idxs)

  dims1 = random.choices(allowed_dims, k = n1)
  dims2 = random.choices(allowed_dims, k = n2)

  nw = np.random.randint(0, min(n1, n2) + 1)

  w1 = random.sample(list(np.arange(0, n1)), nw)
  w2 = random.sample(list(np.arange(0, n2)), nw)

  for i1, i2 in zip(w1, w2):
    dims1[i1] = dims2[i2]

  ws = zip(w1, w2)

  return tuple(dims1), tuple(dims2), list(ws)

random_dims_and_wires(5, [2, 3])

In [ ]:
dims1, dims2, ws = random_dims_and_wires(5, [2, 3])
con = make_contraction(random_tensor(dims1), random_tensor(dims2), ws)

con

In [ ]:
dims = (2, 3)
id = np.eye(np.prod(dims)).reshape(dims + dims)
print(id)

shape = (3, 3, 2, 4, 2)
idxs = [0, 2, 3]

tuple(map(shape.__getitem__, idxs))

In [ ]:
swapt = np.zeros((2, 2, 2, 2))
swapt[0, 0, 0, 0] = swapt[0, 1, 1, 0] = swapt[1, 0, 0, 1] = swapt[1, 1, 1, 1] = 1
swapm = np.reshape(swapt, (4, 4))

U, S, Vh = np.linalg.svd(swapm)
S = np.diag(S)
A = U @ S
B = Vh

aT = np.reshape(A, (2, 2, 2, 2))
bT = np.reshape(B, (2, 2, 2, 2))

aT = np.einsum("klij", aT)
bT = np.einsum("ijkl", bT)

aT

In [ ]:
m = np.arange(0, 128).reshape((16, 8))
u, s, v = np.linalg.svd(m)
s

In [ ]:
import numpy as np

a = np.arange(0, 64).reshape((8, 8))
b = np.arange(0, 64).reshape((8, 8)).T

a @ b

In [ ]:
from math import isqrt
def factors(n):    # (cf. https://stackoverflow.com/a/15703327/849891)
  j = 2
  while n > 1:
    for i in range(j, isqrt(n) + 1):
      if n % i == 0:
        n //= i
        j = i
        yield i
        break
    else:
      if n > 1:
        yield n
        break

for i in range(2, 51):
  print(i, ": ", list(factors(2 ** i - 1)))

In [ ]:
import numpy as np

a = np.array([ 
  -1.71907, -0.211628, 0, 0, 0.992508, 0.122183, 0, 0, 
  0.211628, -1.71907, 0, 0, -0.122183, 0.992508, 0, 0, 
  0, 0, 1.73205, 0, 0, 0, -1, 0, 
  0, 0, 0, 1.73205, 0, 0, 0, -1, 

  -1.71907, -0.211628, 0, 0, -0.992508, -0.122183, 0, 0, 
  0.211628, -1.71907, 0, 0, 0.122183, -0.992508, 0, 0, 
  0, 0, 1.73205, 0, 0, 0, 1, 0, 
  0, 0, 0, 1.73205, 0, 0, 0, 1,


  0, 0, 0, 0, 0, 0, 0, 0, 
  0, 0, 0, 0, 0, 0, 0, 0, 
  0, 0, 0, 0, 0, 0, 0, 0, 
  0, 0, 0, 0, 0, 0, 0, 0, 

  0, 0, 0, 0, 0, 0, 0, 0, 
  0, 0, 0, 0, 0, 0, 0, 0, 
  0, 0, 0, 0, 0, 0, 0, 0, 
  0, 0, 0, 0, 0, 0, 0, 0,


  0, 0, 0, 0, 0, 0, 0, 0, 
  0, 0, 0, 0, 0, 0, 0, 0, 
  0, 0, 0, 0, 0, 0, 0, 0, 
  0, 0, 0, 0, 0, 0, 0, 0, 

  0, 0, 0, 0, 0, 0, 0, 0, 
  0, 0, 0, 0, 0, 0, 0, 0, 
  0, 0, 0, 0, 0, 0, 0, 0, 
  0, 0, 0, 0, 0, 0, 0, 0,


  1.73205, 0, 0, 0, -1, 0, 0, 0, 
  0, 1.73205, 0, 0, 0, -1, 0, 0, 
  0, 0, 1.73205, 0, 0, 0, -1, 0, 
  0, 0, 0, 1.73205, 0, 0, 0, -1, 

  1.73205, 0, 0, 0, 1, 0, 0, 0, 
  0, 1.73205, 0, 0, 0, 1, 0, 0, 
  0, 0, 1.73205, 0, 0, 0, 1, 0, 
  0, 0, 0, 1.73205, 0, 0, 0, 1
])

a = a.reshape((2, 2, 2, 4, 2, 4))
# a = a.transpose((1, 4, 0, 3, 2, 5))
a = a.transpose((2, 0, 3, 4, 1, 5))
a = a.reshape((16, 16))

u, s, v = np.linalg.svd(a)
u @ np.diag(s) @ v
s

In [ ]:
np.random.seed(1)
b = np.int32(np.random.random(256) * 100) / 100
b

In [ ]:
b = b.reshape((2, 2, 2, 4, 2, 4))
# b = b.transpose((1, 4, 0, 3, 2, 5))
b = b.transpose((2, 0, 3, 4, 1, 5))
b = b.reshape((16, 16))

u, s, v = np.linalg.svd(b)
u @ np.diag(s) @ v

print(s)

In [ ]:
# Reference to P2OF output distribution, generated by ChatGPT. 
import numpy as np
import matplotlib.pyplot as plt

# Define constants
n = 5  # number of qubits in the upper register
N = 2 ** n  # number of possible measurement outcomes
r = 7  # period of a^x mod N (here, a=8, N=15)
k_values = list(range(r))  # eigenphase numerators

# Function to compute probability for each y
def probability_y(y):
    prob = 0
    for k in k_values:
        phase = k / r
        sum_exp = sum(np.exp(2j * np.pi * (phase - y / N) * x) for x in range(N))
        prob += abs(sum_exp / N) ** 2
    return prob / len(k_values)

# Compute probabilities for all 16 outcomes
probabilities = [probability_y(y) for y in range(N)]


# Plot the probabilities
plt.figure(figsize=(10, 5))
plt.bar(range(N), probabilities)
plt.xlabel("Measurement outcome (y)")
plt.ylabel("Probability")
plt.title("Probabilities of measuring each output (4-qubit upper register)")
plt.xticks(range(N))
plt.grid(axis='y')
plt.tight_layout()
plt.show()

# Print the raw probabilities
for y, p in enumerate(probabilities):
    print(f"y = {y:2d} ({y:04b}): Probability = {p:.6f}")

In [ ]:
import numpy as np
import scipy

np.matrix([
  [-0.71, 0, 0, 0], 
  [0, 0, 0, 0], 
  [0, 0, 0, 0], 
  [0, 0, 0, 0], 
  [-0.71j, 0, 0, 0], 
  [0, 0, 0, 0], 
  [0, 0, 0, 0], 
  [0, 0, 0, 0]
])

m = np.zeros((16, 32), dtype=complex)
m[0, 0] = 1 / np.sqrt(2)
m[0, 4] = -1 / np.sqrt(2)
m[6, 8] = -1j / np.sqrt(2)
m[6, 12] = -np.exp(2j / 3 * np.pi ) / np.sqrt(2)

q, r, p = scipy.linalg.qr(m, pivoting=True)
print(p)

pm = np.eye(32)[:, p]
r[0:8, :] @ pm.T

r

In [ ]:
m = np.zeros((4, 8), dtype=complex)
m[0, 0] = 1 / np.sqrt(2)
m[0, 2] = -1 / np.sqrt(2)
m[3, 4] = -1j / np.sqrt(2)
m[3, 6] = np.exp(2j / 3 * np.pi ) / np.sqrt(2)

q, r, p = scipy.linalg.qr(m, pivoting=True)
pm = np.eye(8)[:, p]
q @ r

In [ ]:
import numpy as np

def swap_seq(n, swaps, reverse = False):
  xs = np.arange(1, n + 1)
  es = list(enumerate(swaps))
  if (reverse):
    es = reversed(es)

  for (i, j) in es:
    xs[i], xs[j - 1] = xs[j - 1], xs[i]
  return xs

swap_seq(8, [1, 3, 2, 6, 5, 4, 6, 6])
swap_seq(4, [1, 2, 1, 2])

In [ ]:
m = np.zeros((4, 4), dtype=complex)
m[0, 0] = 1
m[0, 1] = 2
m[3, 2] = 3
m[3, 3] = 4

q, r, p = scipy.linalg.qr(m, pivoting=True)
pm = np.eye(4)[:, p]

q @ r

q, r, p = scipy.linalg.qr(q @ r, pivoting=True)
q @ r

In [ ]:
import copy

def bubble_sort(pt):
  res = []
  sorted = False

  while not sorted:
    sorted = True
    for i in range(len(pt) - 1):
      if (pt[i] > pt[i + 1]):
        sorted = False
        res.append(i)
        pt[i], pt[i + 1] = pt[i + 1], pt[i]
  
  return res

# m = 9
# n = 6

# pt = []
# for i in range(m):
#   for j in range(n):
#     pt.append(j * m + i)

# pt = [0, 7, 8, 9, 6, 1, 2, 5, 10, 11, 4, 3]
# pt = [0, 4, 8, 1, 5, 9, 2, 6, 10, 3, 7, 11]
# pt = [0, 7, 8, 1, 6, 9, 2, 5, 10, 3, 4, 11]
# pt = [1, 6, 11, 16, 2, 7, 12, 17, 3, 8, 13, 18, 4, 9, 14, 19, 5, 10, 15, 20]
# pt = [1, 3, 2, 4]

pt = [
               16, 
            9, 17, 25, 
        4, 10, 18, 26, 34, 
     1, 5, 11, 19, 27, 35, 
  0, 2, 6, 12, 20, 28, 36, 42, 47, 
     3, 7, 13, 21, 29, 37, 43, 48, 51, 
        8, 14, 22, 30, 38, 44, 49, 52, 
           15, 23, 31, 39, 45, 50, 
               24, 32, 40, 46, 
                   33, 41
]
pt_copy = copy.deepcopy(pt)
swaps = bubble_sort(pt)

print(swaps)
print(len(swaps))

for i in swaps:
  pt_copy[i], pt_copy[i + 1] = pt_copy[i + 1], pt_copy[i]

print(pt_copy)

In [ ]:
import numpy as np

d = 128
m = np.random.rand(d, d) + 1j * np.random.rand(d, d)
_, s, _ = np.linalg.svd(m)
s